# 7/7 - I. Exports
UI json and LLM json (include channel / line / registry / clip index), size and cost check.

*Original sections: sections 20, 21.*  Barca (team 0) vs Atletico (team 1), first half.

**Run the notebooks in order** (each one reads what the earlier ones saved in `new_cache/pressure_analysis/state/`):

| # | notebook | what it covers |
|---|---|---|
| 1 | 01_pressure_signal.ipynb | B. The pressure signal |
| 2 | 02_where_zone_channel_line.ipynb | C. WHERE - zone, channel, defensive line, shape |
| 3 | 03_timing_and_possession.ipynb | D. WHEN - reaction speed and possession-level pressure |
| 4 | 04_passes_outcomes_triggers.ipynb | E/F. EFFECT and TRIGGERS |
| 5 | 05_summary_and_registry.ipynb | G/H. Summary + event registry |
| 6 | 06_clips.ipynb | H. Clips |
| 7 | **07_exports.ipynb** | I. Exports |

Shared setup (paths, tables, data contract, team-label orientation, helpers) lives in `pressure_common.py` - keep it in the same folder as the notebooks.

In [ ]:
from pressure_common import *

In [ ]:
# handed over by the earlier notebooks:
#   02_where_zone_channel_line: BACK_N, FRONT_N, WIDTH_IQR_MULT, channel_pct, defensive_line_events, line_by_ball_channel, line_by_ball_zone, line_last1_clean, line_last3_clean, pressure_points_clean, team_shape, team_shape_pressure, zone_channel_pct, zone_counts, zone_pct
#   06_clips: CLIP_INDEX
#   01_pressure_signal: CLOSING_SPEED_THRESH_MPS, CLOSING_WINDOW_FRAMES, FRAC_TIGHT_BLOCKED_THRESH, LINE_HEIGHT_DEFENDERS_N, N_PRESSING_THRESH, PRESSURE_DIST_THRESH_M, STEP_OUT_MARGIN_M, STOPPAGE_GAP_SEC, TIGHT_MARK_DIST_M, match_duration_sec, out_of_possession_seconds, stoppages, total_pressure_seconds_by_team
#   05_summary_and_registry: EVENTS, team_pressing_profile
#   04_passes_outcomes_triggers: TRIGGER_COLS, disruption_rate, outcome_pct, pass_summary, passes_from_scratch, ppda_by_team, ppda_detail, reception_triggers, zone_summary
#   03_timing_and_possession: ZONE_WEIGHTS, final_pressure_by_team, possession_events, pressing_intensity_timeline, regain_summary, summary, team_pressure_events, turnovers_annotated, weighted_pressure_seconds
load_state(globals(), [
    'BACK_N',
    'CLIP_INDEX',
    'CLOSING_SPEED_THRESH_MPS',
    'CLOSING_WINDOW_FRAMES',
    'EVENTS',
    'FRAC_TIGHT_BLOCKED_THRESH',
    'FRONT_N',
    'LINE_HEIGHT_DEFENDERS_N',
    'N_PRESSING_THRESH',
    'PRESSURE_DIST_THRESH_M',
    'STEP_OUT_MARGIN_M',
    'STOPPAGE_GAP_SEC',
    'TIGHT_MARK_DIST_M',
    'TRIGGER_COLS',
    'WIDTH_IQR_MULT',
    'ZONE_WEIGHTS',
    'channel_pct',
    'defensive_line_events',
    'disruption_rate',
    'final_pressure_by_team',
    'line_by_ball_channel',
    'line_by_ball_zone',
    'line_last1_clean',
    'line_last3_clean',
    'match_duration_sec',
    'out_of_possession_seconds',
    'outcome_pct',
    'pass_summary',
    'passes_from_scratch',
    'possession_events',
    'ppda_by_team',
    'ppda_detail',
    'pressing_intensity_timeline',
    'pressure_points_clean',
    'reception_triggers',
    'regain_summary',
    'stoppages',
    'summary',
    'team_pressing_profile',
    'team_pressure_events',
    'team_shape',
    'team_shape_pressure',
    'total_pressure_seconds_by_team',
    'turnovers_annotated',
    'weighted_pressure_seconds',
    'zone_channel_pct',
    'zone_counts',
    'zone_pct',
    'zone_summary',
])

## 20. Export for a UI app
Everything above stays as analysis; this section pulls the *finished* numbers computed throughout the notebook into one JSON file a front-end can fetch directly, without re-running the pipeline. Massive per-frame and per-player tables (`off_ball_pressure`, `team_pressure_full`, `passes_from_scratch`'s full row set, raw pressure-heatmap points) are either left out or aggregated first -- e.g. the pressure heatmap is binned into a small grid instead of exporting every point -- since a UI only needs the finished shape of the data, not the frame-by-frame data behind it.

Three cells: (1) helpers + team-level summary tables, (2) event-level tables (pressure events, possession events, stoppages, turnovers), (3) the pressure heatmap grid + writing the JSON file.

In [ ]:
import json

# export location now comes from the Setup cell (EXPORT_DIR = new_cache/pressure_analysis)
UI_EXPORT_PATH = EXPORT_DIR / "pressure_analysis_export.json"

def _df(df, index_name=None):
    """DataFrame/Series -> list of JSON-safe records (NaN -> null, numpy types -> native)."""
    d = df.reset_index()
    if index_name is not None:
        # the restored index is always the FIRST column after reset_index(), whatever its
        # own .name was ("index" if unnamed, or something else if the index was named --
        # e.g. team_pressing_profile's index is already named "team", not "index", so a
        # rename keyed on the literal string "index" silently did nothing for that table)
        d = d.rename(columns={d.columns[0]: index_name})
    return json.loads(d.to_json(orient="records"))

def _num(x):
    """scalar (numpy or python) -> JSON-safe float/int, NaN -> null."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    return x

ui_export = {"meta": {}, "team": {}, "events": {}}

ui_export["meta"] = {
    "team_names": TEAM_NAMES,
    "pitch_length_m": PITCH_LENGTH,
    "pitch_width_m": PITCH_WIDTH,
    "fps": FPS,
    "match_duration_sec": _num(match_duration_sec),
    "stoppage_count": int(len(stoppages)),
    "stoppage_total_sec": _num(stoppages["duration_sec"].sum()),
}

# recompute the small inline percentages that section 4 / 9 only ever printed, never stored
pressing_intensity_pct = {
    team_id: _num(total_pressure_seconds_by_team.get(team_id, 0) / out_of_possession_seconds[team_id] * 100)
    for team_id in [0, 1]
}
zone_weighted_pct_of_max = {
    team_id: _num(
        weighted_pressure_seconds.get(team_id, 0)
        / (out_of_possession_seconds[team_id] * ZONE_WEIGHTS["attacking_third"])
        * 100
    )
    for team_id in [0, 1]
}

line_height_summary = {
    "last1": _df(line_last1_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]), "team"),
    "last3": _df(line_last3_clean.groupby("team")["line_height_m"].agg(["mean", "median", "std"]), "team"),
}

shape_summary = _df(
    team_shape.groupby("team")[["depth_m", "width_m", "compactness_area_m2"]].agg(["mean", "median", "std"]).round(2),
    "team",
)
shape_by_pressure_state = _df(
    team_shape_pressure.groupby(["team", "under_team_pressure"])["compactness_area_m2"].agg(["mean", "median", "count"]).round(2)
)

ui_export["team"] = {
    # the master per-team table -- volume, PPDA, regain rate, turnover rate by zone, counter-press rate
    "pressing_profile": _df(team_pressing_profile, "team_name"),
    "pressing_intensity_pct": pressing_intensity_pct,
    "zone_pressure_counts": _df(zone_counts, "team"),
    "zone_pressure_pct": _df(zone_pct.round(1), "team"),
    "zone_weighted_pct_of_max": zone_weighted_pct_of_max,
    "defensive_line_height": line_height_summary,
    "shape_summary": shape_summary,
    "shape_by_pressure_state": shape_by_pressure_state,
    "ppda": {int(k): _num(v) for k, v in ppda_by_team.items()},
    "ppda_detail": {int(k): v for k, v in ppda_detail.items()},
    "final_pressure_by_team": _df(final_pressure_by_team.round(2), "team"),
    "pass_summary": _df(pass_summary.round(1)),
    "outcome_pct": _df(outcome_pct.round(1)),
    "disruption_rate_pct": {int(k): _num(v) for k, v in disruption_rate.round(1).to_dict().items()},
    "zone_outcome_summary": _df(zone_summary),
    "regain_summary": _df(regain_summary, "team"),
    "counter_press_response_summary": _df(summary),
    "reception_trigger_rates_pct": _df((reception_triggers.groupby("team")[TRIGGER_COLS + ["press_followed"]].mean() * 100).round(1), "team"),
    "pressing_intensity_timeline": _df(pressing_intensity_timeline.round(1), "bin_label"),
}

# --- structured dimensions added in v31: channel, zone x channel grid, defensive-line structure
ui_export["team"]["channel_pressure_pct"] = _df(channel_pct.round(1), "team")
ui_export["team"]["zone_channel_pressure_pct"] = _df(zone_channel_pct.round(1))
ui_export["team"]["defensive_line_by_ball_zone"] = _df(line_by_ball_zone)
ui_export["team"]["defensive_line_by_ball_channel"] = _df(line_by_ball_channel)

print("team-level sections assembled:", list(ui_export["team"].keys()))

In [ ]:
# event-level tables -- moderate row counts (tens to low hundreds), suited to a UI timeline / list view

def _events_with_time(df, frame_cols=("start_frame", "end_frame")):
    out = df.copy()
    for c in frame_cols:
        if c in out.columns:
            out[c.replace("frame", "sec")] = out[c] / FPS
    return out

pressure_events_export = _events_with_time(
    team_pressure_events[
        ["team", "start_frame", "end_frame", "duration_sec", "possession_team",
         "mean_n_pressing", "max_n_pressing", "mean_frac_tight_blocked", "regained_ball"]
    ]
)

possession_events_export = _events_with_time(
    possession_events[
        ["event_id", "possession_team", "pressing_team", "start_frame", "end_frame",
         "duration_sec", "pct_frames_under_pressure"]
    ]
)

stoppages_export = _events_with_time(stoppages[["start_frame", "end_frame", "duration_sec"]])

turnovers_export = turnovers_annotated[
    ["losing_team", "loss_frame", "gain_frame", "time_to_press_sec", "press_response", "regained_within_window"]
].copy()
turnovers_export["loss_sec"] = turnovers_export["loss_frame"] / FPS
turnovers_export["gain_sec"] = turnovers_export["gain_frame"] / FPS

# passes trimmed to the fields a UI pass-map/list would actually use -- not every intermediate column
_pass_cols = ["passer_team", "release_frame", "release_x", "outcome", "under_pressure",
              "pass_distance_m", "progression_m", "pass_outcome_detail"]
if "release_y" in passes_from_scratch.columns:
    _pass_cols.insert(3, "release_y")
passes_export = passes_from_scratch[_pass_cols].copy()
passes_export["release_sec"] = passes_export["release_frame"] / FPS

ui_export["events"] = {
    "pressure_events": _df(pressure_events_export),
    "possession_events": _df(possession_events_export),
    "stoppages": _df(stoppages_export),
    "turnovers": _df(turnovers_export),
    "passes": _df(passes_export),
}

ui_export["events"]["defensive_line_events"] = _df(_events_with_time(defensive_line_events))
ui_export["events"]["event_registry"] = _df(EVENTS.drop(columns="payload"))
if "CLIP_INDEX" in globals():
    ui_export["events"]["clips"] = _df(CLIP_INDEX)      # file names are relative to pressure_analysis/clips/

print("event-level tables assembled:", {k: len(v) for k, v in ui_export["events"].items()})

In [ ]:
# pressure heatmap, binned into a small grid instead of exporting every raw point (Section 11)
HEATMAP_BIN_M = 5.0
x_bins = np.arange(0, PITCH_LENGTH + HEATMAP_BIN_M, HEATMAP_BIN_M)
y_bins = np.arange(0, PITCH_WIDTH + HEATMAP_BIN_M, HEATMAP_BIN_M)

heatmap_by_team = {}
for team_id in [0, 1]:
    sub = pressure_points_clean[pressure_points_clean["team"] == team_id]
    counts, _, _ = np.histogram2d(sub["press_x"], sub["press_y"], bins=[x_bins, y_bins])
    pct_of_team_total = counts / counts.sum() * 100 if counts.sum() > 0 else counts
    cells = []
    for i in range(len(x_bins) - 1):
        for j in range(len(y_bins) - 1):
            if counts[i, j] > 0:
                cells.append({
                    "x_min": float(x_bins[i]), "x_max": float(x_bins[i + 1]),
                    "y_min": float(y_bins[j]), "y_max": float(y_bins[j + 1]),
                    "count": int(counts[i, j]),
                    "pct_of_team_total": round(float(pct_of_team_total[i, j]), 2),
                })
    heatmap_by_team[int(team_id)] = cells

ui_export["team"]["pressure_heatmap_grid"] = heatmap_by_team

with open(UI_EXPORT_PATH, "w") as f:
    json.dump(ui_export, f, indent=2)

print(f"wrote {UI_EXPORT_PATH}")
print("top-level keys:", list(ui_export.keys()))
print("team-level keys:", list(ui_export["team"].keys()))
print("event-level keys:", list(ui_export["events"].keys()))

## 21. Export for an LLM narrative layer
Section 20's `ui_export` stays exactly as-is -- it's the Streamlit UI's contract, and changing its shape would break the app. This section builds a **separate** file for feeding a downstream LLM. Same underlying numbers, but with what a model actually needs to produce real tactical analysis instead of paraphrasing numbers back:

- a `methodology` block spelling out exactly what each threshold/definition means (the pressure gate, PPDA zone convention, possession segmentation, defensive-line and team-shape conventions), pulled live from this notebook's own constants so it can't drift out of sync
- a `data_quality` block with the pipeline's known caveats -- ball-carrier coverage and player-identity fragmentation -- computed live from the data, not hardcoded, so it stays accurate on every rerun
- every team reference made consistently name-readable (`Barca`/`Atletico`), not a mix of `0`/`1` ids and names across tables -- handled by one generic recursive pass so it covers every table without hand-rewriting each one
- human-readable `M:SS` alongside every raw seconds value
- a curated `notable_events` section (longest press per team, fastest/slowest counter-presses, most heavily pressured possessions) so the model has concrete moments to build a narrative around instead of only aggregates

Sending this file to an LLM is implemented in a separate notebook -- this section only prepares and writes it.


In [ ]:
# --- meta: methodology + live-computed data-quality caveats ---

def _mmss(sec):
    """seconds -> "M:SS" string, None-safe."""
    if sec is None or (isinstance(sec, float) and np.isnan(sec)):
        return None
    total = int(round(sec))
    m, s = divmod(total, 60)
    return f"{m}:{s:02d}"

def _add_mmss(df, sec_cols):
    """copy of df with a '<col minus _sec>_mmss' column added next to each seconds column given."""
    out = df.copy()
    for c in sec_cols:
        if c in out.columns:
            out[c.replace("_sec", "_mmss")] = out[c].apply(_mmss)
    return out

TEAM_ID_FIELD_NAMES = {"team", "passer_team", "losing_team", "possession_team", "pressing_team", "carrier_team"}

def _is_team_id(x):
    try:
        return int(x) in TEAM_NAMES
    except (TypeError, ValueError):
        return False

def _llm_readable(obj):
    """Deep-walk an already-JSON-safe structure (dicts/lists/scalars) and make every team
    reference name-readable, in one generic pass instead of hand-fixing each table:
      - a dict whose keys are ALL team ids (e.g. {0: ..., 1: ...}) gets its keys replaced by
        team names (this is what fixes ppda / pressing_intensity_pct / heatmap_by_team etc.,
        which are keyed by raw team_id while pressing_profile is keyed by team name)
      - any dict with a known team-id field (e.g. "team": 0, "passer_team": 1) gets a sibling
        "<field>_name" added
    Nothing is deleted -- every original numeric id stays in place for exact joins."""
    if isinstance(obj, dict):
        keys = list(obj.keys())
        if keys and all(_is_team_id(k) for k in keys):
            return {TEAM_NAMES[int(k)]: _llm_readable(v) for k, v in obj.items()}
        new = {k: _llm_readable(v) for k, v in obj.items()}
        for field in TEAM_ID_FIELD_NAMES:
            if field in new and _is_team_id(new[field]):
                new[f"{field}_name"] = TEAM_NAMES[int(new[field])]
        return new
    if isinstance(obj, list):
        return [_llm_readable(x) for x in obj]
    return obj

# computed live (not hardcoded) so these can't go stale on a rerun with different source data
ball_carrier_coverage_pct = round(float(ball_frame_table["carrier_track_id"].notna().mean() * 100), 1)
n_distinct_player_tracks = int(player_frame_table.loc[player_frame_table["role"] == "player", "track_id"].nunique())

llm_export = {"meta": {}, "team": {}, "events": {}}

llm_export["meta"] = {
    "team_names": TEAM_NAMES,
    "pitch_length_m": PITCH_LENGTH,
    "pitch_width_m": PITCH_WIDTH,
    "fps": FPS,
    "match_duration_sec": _num(match_duration_sec),
    "match_duration_mmss": _mmss(match_duration_sec),
    "stoppage_count": int(len(stoppages)),
    "stoppage_total_sec": _num(stoppages["duration_sec"].sum()),
    "methodology": {
        "pressure_definition": (
            f"an off-ball defender counts as pressuring the ball carrier when within "
            f"{PRESSURE_DIST_THRESH_M}m AND at least one of: touch-tight (<{TIGHT_MARK_DIST_M}m), "
            f"closing speed >= {CLOSING_SPEED_THRESH_MPS} m/s over roughly "
            f"{CLOSING_WINDOW_FRAMES / FPS:.1f}s, or stepped out >= {STEP_OUT_MARGIN_M}m ahead of "
            f"their team's own out-of-possession baseline defensive line. Raw proximity alone is "
            f"NOT treated as pressure -- this is meant to exclude a passive deep block that the "
            f"attacker simply walked into."
        ),
        "possession_segmentation": (
            "possession events are segmented from carrier_track_id transitions with keep-last gap "
            f"fill; a no-carrier gap >= {STOPPAGE_GAP_SEC}s is treated as a stoppage rather than a "
            "possession change."
        ),
        "ppda_convention": "PPDA zone is classified relative to the DEFENDING team's own third, not the attacking team's.",
        "defensive_line_convention": (
            f"defensive line height = average distance from own goal of the last-"
            f"{LINE_HEIGHT_DEFENDERS_N} defenders, computed only on out-of-possession frames."
        ),
        "team_shape_convention": (
            f"depth = distance between the back-{BACK_N} and front-{FRONT_N} outfield players "
            f"along the team's own attack direction; width = IQR-trimmed lateral spread "
            f"(multiplier {WIDTH_IQR_MULT}); both restricted to out-of-possession frames."
        ),
        "channel_convention": (
            "left / centre / right are measured in the PRESSING (or defending) team's own frame: the team attacks towards +x, "
            "left = its left-hand side. Thirds are the same convention (own / middle / attacking relative to that team). "
            "zone_channel_pressure_pct is the share of the team's own pressure frames falling in each of the 9 cells."
        ),
        "defensive_line_structure": (
            "defensive_line_by_ball_zone / _by_ball_channel use defending frames only; zone and channel refer to where the BALL is "
            "relative to the defending team. tilt_m = mean height of the back-4 players on the left half minus the right half "
            "(positive = left side of the line is higher). defensive_line_events are sustained high / deep line periods (team's own "
            "80th / 20th percentile) and step-ups of >= 8 m within 2 s."
        ),
        "event_registry_note": (
            "event_registry lists every metric's events in one schema (family, subtype, focus team, frames, zone, channel, target, score); "
            "'team' is always the pressing / defending (focus) team. clips lists the annotated video clips rendered for the best events."
        ),
        "stoppage_handling": (
            f">= {STOPPAGE_GAP_SEC}s continuous no-carrier gaps are excluded from live-play stats "
            "as stoppages (injury, goal, VAR check, cooling break, etc.)."
        ),
        # the four entries below were added after an LLM report flagged apparent inconsistencies
        # in the exported tables; each was individually checked against the actual computation
        # cells -- most turned out to be different metrics that were never meant to match, not
        # data bugs, but the naming/scope was ambiguous enough to cause a real misread. Documented
        # here rather than "fixed" since the underlying numbers were already correct.
        "event_count_note": (
            "counts named 'n_events' or similar mean different things in different tables and "
            "will not match each other: regain_summary counts individual pressure EVENTS (a team "
            "can have several per possession, or one spanning a possession boundary), while "
            "final_pressure_by_team's n_events counts POSSESSION events instead. A 144-vs-101 "
            "style mismatch between these is expected, not a data error."
        ),
        "zone_outcome_summary_convention": (
            "zone_outcome_summary's 'zone' column is relative to the team the row's actions "
            "belong to (the team being pressured), not the pressing team -- e.g. team=1's "
            "own_third row is about team 1's actions in team 1's own third, and credit for any "
            "disruption/turnover there belongs to team 0's press. pressing_profile's "
            "turnover_rate_<zone> columns are already re-oriented to the PRESSING team's own "
            "zone labels, so the two tables use zone labels from opposite perspectives by design."
        ),
        "pass_summary_vs_outcome_pct_scope": (
            "pass_summary's success_rate is completed/attempted PASSES only. outcome_pct covers a "
            "broader event set (passes + tackles + long-ball-gap events) split into finer outcome "
            "categories (progressed/forced-sideways/forced-backward/lost-intercepted/etc). The two "
            "tables have different denominators and category schemes by design -- e.g. "
            "pass_summary success % and (100 - outcome_pct's lost-category %) will not reconcile, "
            "and that is expected, not an error."
        ),
        "lane_blocking_only_pressure_events": (
            "a team-pressure frame is flagged under_team_pressure if n_pressing >= "
            f"{N_PRESSING_THRESH} OR frac_tight_blocked >= {FRAC_TIGHT_BLOCKED_THRESH} (passing-"
            "lane blocking alone can qualify, without any proximity presser meeting the "
            "closing-speed/touch-tight/step-out gate from the pressure_definition above). A "
            "pressure event can therefore legitimately show mean_n_pressing and max_n_pressing "
            "near zero throughout if it was driven entirely by lane-blocking rather than "
            "proximity pressure -- check mean_frac_tight_blocked on that event before treating a "
            "low-n_pressing event as a data error."
        ),
        "shape_by_pressure_state_orientation": (
            "shape_by_pressure_state's under_team_pressure reads as 'this team is being "
            "pressed' but means the OPPOSITE: tpf['team'] (the source of under_team_pressure) "
            "is the team APPLYING pressure, so a True row is this team's own shape WHILE THEY "
            "ARE PRESSING, not their shape while under the opponent's press. Confirmed by frame "
            "counts: the True rows match each team's own zone_pressure_counts totals exactly."
        ),
        "disruption_rate_pct_orientation": (
            "disruption_rate_pct uses the SAME orientation as zone_outcome_summary -- team = the "
            "team whose action was disrupted, credit for the disruption belongs to the OPPONENT's "
            "press (see zone_outcome_summary_convention above). It is computed once, directly, "
            "from unrounded event-level outcomes -- reconstructing it by averaging "
            "zone_outcome_summary's already-1-decimal-rounded per-zone rates will drift by a few "
            "tenths of a point from double-rounding; that drift is expected numerical noise, not "
            "an orientation error."
        ),
    },
    "data_quality": {
        "ball_carrier_coverage_pct": ball_carrier_coverage_pct,
        "carrier_coverage_note": (
            "the ball carrier is only identified for this share of frames; frames without a "
            "carrier are excluded from carrier-dependent metrics (pressure, passes, PPDA), which "
            "slightly undercounts fast or contested play."
        ),
        "distinct_player_track_ids": n_distinct_player_tracks,
        "identity_fragmentation_note": (
            "player identities are tracked, not fully re-identified across the whole match, so "
            "this count is higher than the true number of players (camera-angle / occlusion "
            "fragmentation). Player-level stats should be read as per-track-segment, not fully "
            "resolved per-athlete; team-level aggregates are not affected by this."
        ),
    },
}

print("LLM export meta assembled:", list(llm_export["meta"].keys()))


In [ ]:
# --- team-level tables, rekeyed consistently to team names via the generic walker above ---

llm_export["team"] = _llm_readable(ui_export["team"])
print("LLM export team-level keys:", list(llm_export["team"].keys()))

# --- event-level tables: reuse section 20's per-row frames, add mm:ss + name-readable teams ---

_sec_cols = {
    "pressure_events": ["start_sec", "end_sec"],
    "possession_events": ["start_sec", "end_sec"],
    "stoppages": ["start_sec", "end_sec"],
    "turnovers": ["loss_sec", "gain_sec", "time_to_press_sec"],
    "passes": ["release_sec"],
}
_source_dfs = {
    "pressure_events": pressure_events_export,
    "possession_events": possession_events_export,
    "stoppages": stoppages_export,
    "turnovers": turnovers_export,
    "passes": passes_export,
}

llm_events = {}
for _name, _df_src in _source_dfs.items():
    _enriched = _add_mmss(_df_src, _sec_cols[_name])
    llm_events[_name] = _llm_readable(_df(_enriched))

if "CLIP_INDEX" in globals():
    llm_events["clips"] = _llm_readable(_df(CLIP_INDEX))

llm_export["events"] = llm_events
print("LLM export event-level counts:", {k: len(v) for k, v in llm_events.items()})


In [ ]:
# --- notable_events: a curated set of concrete moments for the LLM to anchor a narrative on ---

def _top(df, col, n, ascending=False):
    return df.sort_values(col, ascending=ascending, na_position="last").head(n)

notable = {}

notable["longest_pressure_events_by_team"] = {
    TEAM_NAMES[tid]: _llm_readable(_df(_add_mmss(
        _top(pressure_events_export[pressure_events_export["team"] == tid], "duration_sec", 3),
        _sec_cols["pressure_events"],
    )))
    for tid in [0, 1]
}

notable["fastest_counter_presses"] = _llm_readable(_df(_add_mmss(
    _top(turnovers_export, "time_to_press_sec", 5, ascending=True), _sec_cols["turnovers"]
)))
notable["slowest_counter_presses"] = _llm_readable(_df(_add_mmss(
    _top(turnovers_export, "time_to_press_sec", 5, ascending=False), _sec_cols["turnovers"]
)))

notable["most_heavily_pressured_possessions"] = _llm_readable(_df(_add_mmss(
    _top(possession_events_export, "pct_frames_under_pressure", 5), _sec_cols["possession_events"]
)))

llm_export["events"]["notable_events"] = notable

# --- two files, both COMPACT (no indent) -- this JSON is sent over the wire as literal text,
# so pretty-printing it (as section 20 does for the UI file) just burns tokens/cost for
# nothing; indent=2 vs compact is purely cosmetic and inflates size ~50% for zero benefit here.
#
# 1) the SLIM default payload (meta + team aggregates + notable_events, no raw event rows) --
#    this is what a first LLM call should send: the whole tactical picture in a few hundred
#    rows instead of 1,000+.
# 2) the FULL export (same as before, now compact) -- kept as a reserve/lookup file for a
#    follow-up call that needs specific rows (e.g. "show me every pass under pressure"),
#    not something to paste into a prompt wholesale.

llm_default_payload = {
    "meta": llm_export["meta"],
    "team": llm_export["team"],
    "notable_events": notable,
}

LLM_EXPORT_PATH = EXPORT_DIR / "pressure_analysis_llm_export.json"
LLM_EXPORT_DEFAULT_PATH = EXPORT_DIR / "pressure_analysis_llm_default.json"

with open(LLM_EXPORT_PATH, "w") as f:
    json.dump(llm_export, f, separators=(",", ":"))
with open(LLM_EXPORT_DEFAULT_PATH, "w") as f:
    json.dump(llm_default_payload, f, separators=(",", ":"))

print(f"wrote {LLM_EXPORT_PATH} (full, compact)")
print(f"wrote {LLM_EXPORT_DEFAULT_PATH} (default payload, compact)")
print("meta keys:", list(llm_export["meta"].keys()))
print("team keys:", list(llm_export["team"].keys()))
print("event keys:", list(llm_export["events"].keys()))
print("notable_events keys:", list(notable.keys()))


### Size / cost check
Prints the actual compact byte size of both files just written, a rough token estimate (Anthropic's own rule of thumb: ~4 characters per token -- see docs.claude.com/en/docs/about-claude/pricing), and the resulting per-call cost across the three current model tiers, for an assumed output length. Re-run this any time the underlying match data changes instead of guessing -- and double check the PRICING table below against https://platform.claude.com/docs/en/about-claude/pricing since rates do change.


In [ ]:
import os

# pricing as of the last time this cell was checked against
# https://platform.claude.com/docs/en/about-claude/pricing -- $ per million tokens (input, output)
PRICING_PER_MTOK = {
    "Claude Haiku 4.5": (1, 5),
    "Claude Sonnet 5":  (2, 10),
    "Claude Opus 5":    (5, 25),
}
ASSUMED_OUTPUT_TOKENS = 3000  # a detailed multi-section tactical write-up, rough order of magnitude

def _size_report(path, label):
    n_bytes = os.path.getsize(path)
    n_tokens_est = n_bytes / 4  # Anthropic's documented rule of thumb, not an exact tokenizer count
    print(f"{label}: {n_bytes:,} bytes  (~{n_tokens_est:,.0f} input tokens, rough estimate)")
    print(f"{'':>{len(label) + 2}}per-call cost at ~{ASSUMED_OUTPUT_TOKENS:,} output tokens:")
    for model, (in_rate, out_rate) in PRICING_PER_MTOK.items():
        cost = n_tokens_est / 1_000_000 * in_rate + ASSUMED_OUTPUT_TOKENS / 1_000_000 * out_rate
        print(f"    {model:<18} ~${cost:.3f}")
    print()

_size_report(LLM_EXPORT_DEFAULT_PATH, "default payload (meta + team + notable_events)")
_size_report(LLM_EXPORT_PATH, "full export (adds every raw event/pass row)")
